# PI — 노트북 버전

분기 데이터 하나와 첫 origin 날짜를 넣으면, 모델을 고르고 **모든 origin의 예측 분위수(딕셔너리 1)** 와 **통합성적표(딕셔너리 2)** 를 만드는 노트북입니다. 위에서 아래로 실행하면 됩니다.

| 셀 | 종류 | 하는 일 | 바꿀 것 |
|---|---|---|---|
| [1] | Input | 분기 데이터 넣기 | `csv_name` |
| [2] | Input | 첫 origin(Out of Sample 시작) 날짜 넣기 | `first_insample_end_date` |
| [3] | Model Selection | 최적 모델 선정 | |
| [4] | Output | 딕셔너리 1: 모든 origin의 예측 분위수 | |
| [5] | Output | 딕셔너리 2: 통합성적표 | |
| [저장] | 선택 | 결과를 JSON, CSV 파일로 저장 | |

각 셀은 두 종류로 나뉘어 있습니다.

| 셀 | 표시 | 내용 |
|---|---|---|
| **작동 코드** | 셀 첫 줄 `# ▶ 작동 코드`, 태그 `작동코드` | 실제 계산만 있습니다. 이 셀들만 순서대로 실행하면 `dict1`, `dict2`가 만들어집니다. |
| **설명 코드** | 셀 첫 줄 `# ▷ 설명 코드`, 태그 `설명코드` | 결과를 확인하는 `print`, `display`만 있습니다. 없어도 동작하므로 건너뛰거나 지워도 됩니다. |

> **이 노트북만 떼어서 쓰려면**: `▶ 작동 코드` 셀([1]~[5], 저장까지)만 복사해 가면 됩니다. 같은 폴더에 `selection.py`, `ts_models.py`가 있어야 합니다.

## [1] Input — 분기 데이터

`csv_name`에 분기 데이터 CSV 이름을 넣습니다. `data/` 폴더에서 찾으며 `.csv`는 생략해도 됩니다. CSV는 `Date`, `Data` 두 열이고, `Date`는 그 날짜가 속한 분기로 읽습니다(2018-12-01은 2018Q4).
데이터는 최소 28분기가 필요하고, 앞뒤의 빈 값은 잘라 냅니다.

In [1]:
# ▶ 작동 코드 — [1] 입력값: 데이터 CSV 이름 (여기만 바꾸세요)
csv_name = "X1_l1"                  # 입력 데이터 CSV 이름 (.csv는 생략 가능)

In [2]:
# ▶ 작동 코드 — [1] 데이터 읽기와 검사
import sys
sys.path.insert(0, ".")
from pathlib import Path
import numpy as np
import pandas as pd

# CSV 찾기: 현재 폴더, data/ 순서
name = Path(csv_name) if Path(csv_name).suffix else Path(csv_name).with_suffix(".csv")
found = [c for c in (name, Path("data") / name.name) if c.exists()]
if not found:
    raise FileNotFoundError(f"CSV를 찾지 못했다: {csv_name} (현재 폴더, PI_module/data/ 확인)")
path = found[0]
df = pd.read_csv(path, encoding="utf-8-sig")
missing = {"Date", "Data"} - set(df.columns)
if missing:
    raise ValueError(f"CSV에 Date, Data 두 열이 있어야 한다 (없는 열: {', '.join(sorted(missing))}, 있는 열: {', '.join(map(str, df.columns))})")
df = df[["Date", "Data"]]

# 값 검사: 숫자가 아닌 값은 오류, 앞뒤의 빈 값은 잘라 내고 가운데 빈 값은 오류
try:
    dates = pd.to_datetime(df["Date"])
except (ValueError, TypeError) as e:
    raise ValueError(f"Date를 날짜로 읽지 못했다: {e}")
val = pd.to_numeric(df["Data"], errors="coerce")
bad = val.isna() & df["Data"].notna() & (df["Data"].astype(str).str.strip() != "")
if bad.any():
    raise ValueError(f"Data에 숫자가 아닌 값이 있다 (Date {dates[bad.idxmax()].date()}: {df['Data'][bad.idxmax()]!r})")
ok = val.notna()
if not ok.any():
    raise ValueError("Data에 값이 없다")
inner = val[ok.idxmax():ok[::-1].idxmax() + 1]
if inner.isna().any():
    raise ValueError(f"Data의 가운데에 빈 값이 있다 (Date {dates[inner.isna().idxmax()].date()})")

# Date -> 분기 (오름차순으로 연속된 분기여야 한다)
per = pd.PeriodIndex(dates[inner.index], freq="Q")
if len(per) > 1 and not ((per[1:] - per[:-1]).map(lambda x: x.n) == 1).all():
    raise ValueError("Date가 오름차순으로 연속된 분기가 아니다 (순서, 중복, 빠진 분기를 확인하세요)")
q = np.array([f"{p.year}Q{p.quarter}" for p in per])
y = inner.astype(float).to_numpy()

In [3]:
# ▷ 설명 코드 — [1] 읽은 데이터 확인 (없어도 동작)
from IPython.display import display

print(f"{path}: {len(y)}분기 ({q[0]} ~ {q[-1]})")
display(df.head(3))

data\X1_l1.csv: 59분기 (2011Q2 ~ 2025Q4)


,Date,Data
0,2011-06-01,70.103049
1,2011-09-01,-220.877693
2,2011-12-01,158.997366


## [2] Input — 첫 origin 날짜 (Out of Sample 기간 설정)

`first_insample_end_date`에 첫 origin의 마지막 분기가 속한 날짜를 넣습니다(예: `'2018-12-01'`이면 2018Q4까지의 데이터로 첫 예측을 만듭니다). 이 날짜가 속한 분기까지를 훈련에 쓰는 것이 첫 origin입니다. 그 앞쪽(훈련 최소 20분기부터)은 리샘플링 오차를 쌓는 보정 구간이고, 첫 origin부터 데이터 끝까지가 Out of Sample(채점) 구간입니다.
입력 가능한 범위는 `데이터의 24번째 분기 ~ 끝에서 5번째 분기`이며, 벗어나면 오류가 납니다.

In [4]:
# ▶ 작동 코드 — [2] 입력값: 첫 origin 날짜 (여기만 바꾸세요)
first_insample_end_date = "2019-12-01"       # 첫 origin의 마지막 분기가 속한 날짜 (2019-12-01 -> 2019Q4)

In [5]:
# ▶ 작동 코드 — [2] 첫 origin 정하기와 검사
import selection as S                         # 모델 선정 엔진 (같은 폴더의 selection.py)

MIN_CALIB = 4                                # 보정 origin 최소 개수 (오차 풀이 비지 않으려면 h=4를 쌓을 만큼은 필요)

if len(y) < S.MIN_TRAIN + MIN_CALIB + S.H:
    raise ValueError(f"데이터가 너무 짧다: 훈련 {S.MIN_TRAIN} + 보정 {MIN_CALIB} + 정답 {S.H}분기, 최소 {S.MIN_TRAIN + MIN_CALIB + S.H}분기가 필요하다 (현재 {len(y)}분기)")
if np.std(y) == 0:
    raise ValueError("Data 값이 모두 같아서 모델을 적합할 수 없다")
try:
    end = pd.Period(pd.Timestamp(first_insample_end_date), freq="Q")
except ValueError as e:
    raise ValueError(f"첫 origin 날짜를 읽지 못했다: {first_insample_end_date!r} (예: '2019-12-01') | {e}")
lab = f"{end.year}Q{end.quarter}"
if lab not in q:
    raise ValueError(f"첫 origin의 분기 {lab}가 데이터({q[0]}~{q[-1]})에 없다")
o0 = int(np.flatnonzero(q == lab)[0]) + 1    # 첫 origin: 훈련 관측 수
calib = o0 - S.MIN_TRAIN                     # 보정 origin 수
if calib < MIN_CALIB:
    raise ValueError(f"첫 origin이 너무 이르다: 훈련 {S.MIN_TRAIN}분기 + 보정 {MIN_CALIB}분기가 필요해 {q[S.MIN_TRAIN + MIN_CALIB - 1]} 이후여야 한다")
if o0 + S.H > len(y):
    raise ValueError("첫 origin 뒤에 정답이 4분기 이상 있어야 채점할 수 있다")
cfg = S.VariableConfig(code=Path(csv_name).stem, csv="", column="Data", initial_calib=calib)

In [6]:
# ▷ 설명 코드 — [2] 첫 origin과 입력 가능한 범위 확인 (없어도 동작)
print(f"첫 origin: {lab}까지의 데이터({o0}분기)로 예측 시작")
print(f"보정 구간 origin {calib}개, 이후 origin부터 채점 (데이터 끝까지 {len(y) - o0 + 1}개 origin)")
print(f"입력 가능한 날짜 범위: {q[S.MIN_TRAIN + MIN_CALIB - 1]} ~ {q[len(y) - 5]}")

첫 origin: 2019Q4까지의 데이터(35분기)로 예측 시작
보정 구간 origin 15개, 이후 origin부터 채점 (데이터 끝까지 25개 origin)
입력 가능한 날짜 범위: 2017Q1 ~ 2024Q4


## [3] Model Selection — 최적 모델 선정

모델 3종(AR 단독, GARCH(1,1), SV) × (d, D) 4가지 × 스케일(원, 로그) × 구간 방식(자체, 리샘플링)을 모두 백테스트해서, **CRPS 1등의 ±10% 이내 후보 중 커버리지가 90%에 가장 가까운 조합**을 고릅니다.
첫 실행은 몇 분 걸리고, 같은 입력의 두 번째부터는 `cache/`를 읽어 바로 끝납니다.

In [7]:
# ▶ 작동 코드 — [3] Model Selection: 최적 모델 선정
res = S.run_grid(cfg, data=(q, y))            # 모든 조합 백테스트 (캐시 사용)
sel = S.select(res)                          # 선정 규칙: CRPS 1등의 +-10% 이내 중 커버리지가 90%에 가장 가까운 것
m, d, D, sc, iv = sel["선정"]                # 모델, d, D, 스케일, 구간 방식

In [8]:
# ▷ 설명 코드 — [3] 선정 결과와 후보 확인 (없어도 동작)
from IPython.display import display

print(f"선정 모델: {m} (d={d}, D={D}) {sc} 스케일, {iv} 구간 | 후보 {len(sel['후보'])}/{sel['전체 조합']}개")
print("후보 (CRPS 1등의 +-10% 이내, 커버리지가 90%에 가까운 순, 맨 위가 선정):")
display(sel["후보"].round(4))

선정 모델: GARCH (d=0, D=0) 원 스케일, 리샘플링 구간 | 후보 6/24개
후보 (CRPS 1등의 +-10% 이내, 커버리지가 90%에 가까운 순, 맨 위가 선정):


CRPS     MSE     MAE  커버리지(%)       폭
모델    d D 스케일 구간                                           
GARCH 0 0 원   리샘플링  0.7453  1.5448  1.0214  82.1429  4.0807
SV    0 0 원   리샘플링  0.7455  1.5185  1.0083  80.9524  4.0891
              자체    0.7300  1.5185  1.0083  73.8095  3.5539
AR단독  0 0 원   리샘플링  0.7320  1.4963  1.0059  73.8095  3.2787
              자체    0.7285  1.4963  1.0059  71.4286  3.0389
GARCH 0 0 원   자체    0.7390  1.5448  1.0214  70.2381  3.1093

## [4] Output — 딕셔너리 1: 모든 origin의 예측 분위수

선정 모델로 모든 origin의 예측을 만듭니다(같은 시드 42라 선정 때와 같은 난수).
- key: origin의 마지막 분기 (`'2019-Q4'` = 2019-Q4까지의 데이터로 만든 예측, h1은 2020-Q1)
- value: 9 × 4 배열 — 행은 5·10·20·30·50·70·80·90·95% 분위수, 열은 h1~h4 (원래 단위)
- 데이터 끝 이후 4분기를 예측하는 마지막 origin까지 모두 들어 있습니다.

In [9]:
# ▶ 작동 코드 — [4] Output: 딕셔너리 1
LEVELS = (5, 10, 20, 30, 50, 70, 80, 90, 95)  # 분위수(%)

# 선정 조합 하나만 분위수와 함께 다시 계산하고, 선정된 구간 방식의 행만 쓴다
df4, _ = S._run_one(cfg, q, y, m, d, D, sc == "로그", levels=[x / 100 for x in LEVELS])
df4 = df4[df4["구간"] == iv]

dict1 = {}
for o, g in df4.groupby("origin"):
    g = g.sort_values("h")
    dict1[f"{q[o - 1][:4]}-Q{q[o - 1][5]}"] = np.vstack(g["분위수"].to_list()).T      # (9, 4)

In [10]:
# ▷ 설명 코드 — [4] 딕셔너리 1 확인 (없어도 동작)
from IPython.display import display

print(f"딕셔너리 (1): origin {len(dict1)}개 ({list(dict1)[0]} ~ {list(dict1)[-1]}), 값의 모양 {dict1[list(dict1)[0]].shape}")
key = list(dict1)[0]
print(f"dict1[{key!r}] (행: 분위수, 열: 몇 분기 앞)")
display(pd.DataFrame(dict1[key], index=[f"{x}%" for x in LEVELS], columns=["h1", "h2", "h3", "h4"]).round(1))

딕셔너리 (1): origin 25개 (2019-Q4 ~ 2025-Q4), 값의 모양 (9, 4)
dict1['2019-Q4'] (행: 분위수, 열: 몇 분기 앞)


,h1,h2,h3,h4
5%,-151.5,-192.4,-143.0,-150.0
10%,-116.7,-117.9,-134.8,-150.0
20%,17.5,-21.0,-71.2,-79.8
30%,55.1,-0.7,-66.1,-77.4
50%,85.8,49.6,9.6,-5.3
70%,179.9,118.4,112.2,45.6
80%,218.7,211.9,166.5,85.9
90%,297.9,272.2,171.2,147.5
95%,515.0,379.6,243.3,251.6


## [5] Output — 딕셔너리 2: 통합성적표

선정 모델의 통합성적표입니다. 정답이 4분기 모두 있는 origin의 평균이고, `mae`, `mse`, `crps`는 전체 표준편차로 정규화한 값(mae는 |실제 − 점예측| ÷ 표준편차, mse는 그 제곱), `coverage`는 90% 구간(5~95%) 적중률(%)입니다.

In [11]:
# ▶ 작동 코드 — [5] Output: 딕셔너리 2
t = sel["표"].loc[sel["선정"]]
dict2 = {"mae": float(t["MAE"]), "mse": float(t["MSE"]), "crps": float(t["CRPS"]), "coverage": float(t["커버리지(%)"])}

In [12]:
# ▷ 설명 코드 — [5] 딕셔너리 2 확인 (없어도 동작)
dict2

{'mae': 1.0213911377929437,
 'mse': 1.5447907112973662,
 'crps': 0.7452902085978491,
 'coverage': 82.14285714285714}

## [저장] 결과 파일 저장 (선택)

딕셔너리 1, 2를 `output/<csv명>/` 폴더에 **파일 4개**로 저장합니다. 같은 변수는 같은 폴더의 같은 이름 파일을 덮어씁니다.

| 파일 | 내용 |
|---|---|
| `<csv명>_forecast.json` | 딕셔너리 (1) 그대로 |
| `<csv명>_score.json` | 딕셔너리 (2) 그대로 |
| `<csv명>_forecast.csv` | 딕셔너리 (1)을 긴 표로 (origin, h, 예측 분기, p5 ~ p95) |
| `<csv명>_score.csv` | 딕셔너리 (2)를 한 줄 표로 |

In [13]:
# ▶ 작동 코드 — [저장] 결과 파일 저장
import json

stem = Path(csv_name).stem
out = Path("output") / stem
out.mkdir(parents=True, exist_ok=True)

# 딕셔너리 (1)을 긴 표로: origin, h, 예측 분기, p5 ~ p95
rows = []
for k, arr in dict1.items():
    p0 = pd.Period(f"{k[:4]}{k[5:]}", freq="Q")
    for h in range(arr.shape[1]):
        tq = f"{(p0 + h + 1).year}Q{(p0 + h + 1).quarter}"
        rows.append({"origin": k, "h": h + 1, "예측 분기": tq, **{f"p{l}": arr[i, h] for i, l in enumerate(LEVELS)}})

pd.DataFrame(rows).to_csv(out / f"{stem}_forecast.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([dict2]).to_csv(out / f"{stem}_score.csv", index=False, encoding="utf-8-sig")
(out / f"{stem}_forecast.json").write_text(json.dumps({k: v.tolist() for k, v in dict1.items()}, ensure_ascii=False), encoding="utf-8")
(out / f"{stem}_score.json").write_text(json.dumps(dict2, ensure_ascii=False), encoding="utf-8")

113

In [14]:
# ▷ 설명 코드 — [저장] 저장된 파일 확인 (없어도 동작)
from IPython.display import display

print("저장된 파일:", sorted(f.name for f in out.glob(stem + "_*")))
display(pd.read_csv(out / f"{stem}_forecast.csv").head(5).round(1))

저장된 파일: ['X1_l1_forecast.csv', 'X1_l1_forecast.json', 'X1_l1_score.csv', 'X1_l1_score.json']


,origin,h,예측 분기,p5,p10,p20,p30,p50,p70,p80,p90,p95
0,2019-Q4,1,2020Q1,-151.5,-116.7,17.5,55.1,85.8,179.9,218.7,297.9,515.0
1,2019-Q4,2,2020Q2,-192.4,-117.9,-21.0,-0.7,49.6,118.4,211.9,272.2,379.6
2,2019-Q4,3,2020Q3,-143.0,-134.8,-71.2,-66.1,9.6,112.2,166.5,171.2,243.3
3,2019-Q4,4,2020Q4,-150.0,-150.0,-79.8,-77.4,-5.3,45.6,85.9,147.5,251.6
4,2020-Q1,1,2020Q2,-263.8,-259.1,-123.5,-73.9,-22.4,104.0,146.5,233.2,470.9
